# LegalQA Task 2 — Kaggle Dual-T4 CUDA Smoke Gate (Notion DSC 2026)
Lightweight CUDA, VRAM, QLoRA, and Liger-kernel compatibility smoke test launcher.
- **Target Hardware**: Kaggle Dual NVIDIA T4 (GPU 0: Generator | GPU 1: Retrieval/Reranker)
- **Goal**: Validate worst-case sequence probe, short endurance stability, and mini-evaluation before Colab A100 training.
- **Artifact Export**: Outputs  to .

In [ ]:
# Cell 1: Environment & Guardrails
import os, sys

# Disable Transformers 5.0 async load to prevent transient VRAM spikes during 4-bit load on T4
os.environ["HF_DEACTIVATE_ASYNC_LOAD"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
print("Transformers async model loading: DISABLED for T4-safe QLoRA load")
print("PyTorch allocator configured with expandable_segments:True,max_split_size_mb:128")

SEED = 42
CONFIG_PATH = "configs/task2/runtime/kaggle_t4x2.yaml"
LEGACY_CONFIG_PATH = "configs/kaggle_smoke_t4.yaml"
print(f"Config target: {CONFIG_PATH}")


In [ ]:
# Cell 2: Hardware & Device Allocation
import os, sys, subprocess, torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Kaggle GPU execution but torch.cuda.is_available() is False.")

gpu_count = torch.cuda.device_count()
print(f"CUDA GPUs Detected: {gpu_count}")
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB | Compute: sm_{p.major}{p.minor}")

try:
    print("=== Initial NVIDIA-SMI Telemetry ===")
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print(f"nvidia-smi check skipped: {e}")

GEN_DEVICE = "cuda:0"
RETRIEVAL_DEVICE = "cuda:1" if gpu_count >= 2 else "cuda:0"
print(f"Hardware Allocation -> Generator: {GEN_DEVICE} | Retrieval/Reranker: {RETRIEVAL_DEVICE}")


In [ ]:
# Cell 3: Code Root & Workspace Bootstrap
from pathlib import Path
import os, sys, subprocess

code_root = os.path.abspath("LegalQA")
if not os.path.isdir(code_root):
    print("Cloning LegalQA repository from GitHub...")
    subprocess.check_call(["git", "clone", "https://github.com/silent9669/LegalQA.git"])

# Fetch all branches and check out feature branch or candidate
subprocess.check_call(["git", "-C", code_root, "fetch", "origin"])

# Look for mounted candidate manifest
candidate_file = Path("/kaggle/input/legalqa-candidate/candidate_manifest.json")
if not candidate_file.exists():
    cands = list((Path(code_root) / "artifacts" / "candidates").glob("*/candidate_manifest.json"))
    if cands:
        candidate_file = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]

if candidate_file.exists():
    cand_data = json.loads(candidate_file.read_text(encoding="utf-8"))
    cand_sha = cand_data.get("git_commit_sha")
    print(f"Checking out exact candidate SHA: {cand_sha}...")
    subprocess.check_call(["git", "-C", code_root, "checkout", "--detach", cand_sha])
else:
    print("Notice: Checking out feature/harden-task2-five-gates...")
    subprocess.check_call(["git", "-C", code_root, "checkout", "feature/harden-task2-five-gates"])

head_rev = subprocess.check_output(["git", "-C", code_root, "rev-parse", "HEAD"], text=True).strip()
print(f"Active Git Commit: {head_rev}")

if code_root not in sys.path:
    sys.path.insert(0, code_root)
print(f"Active Code Root: {code_root}")


In [ ]:
# Cell 4: Dependency Bootstrap & Runtime Verification
import scripts.bootstrap_kaggle_env as bstrap

bstrap.print_preinstalled_environment()
BOOTSTRAP_RESULT = bstrap.bootstrap_dependencies()
bstrap.verify_runtime_imports(strict=True)
print("Dependency bootstrap complete: all user-space packages verified.")


In [ ]:
# Cell 5: Dataset Mount & Integrity Verification
from src.task2.path_resolver import resolve_runtime_paths
from src.task2.dataset.validator import validate_dataset

paths = resolve_runtime_paths("/kaggle/input", strict=False)
print(f"Resolved Dataset Root: {paths.get('runtime_root')}")

schema_file = os.path.join(code_root, "configs/dataset_schema.yaml")
val_report = validate_dataset(data_dir=paths["runtime_root"], schema_path=schema_file)
print(f"Dataset Manifest Verified: {val_report.get('manifest_verified')} (Status: {val_report.get('status')})")
if val_report.get("status") != "PASS":
    raise RuntimeError(f"Dataset validation failed: {val_report.get('errors')}")


In [ ]:
# Cell 6: Execute GPU Gate Runner
import os, sys, subprocess, gc, torch
from pathlib import Path

# Ensure code_root in sys.path
if code_root not in sys.path:
    sys.path.insert(0, code_root)

# Ensure candidate exists
if not candidate_file.exists():
    print("Freezing candidate on-the-fly from active HEAD...")
    from scripts.freeze_candidate import freeze_candidate
    freeze_candidate(allow_dirty=True)
    cands = list((Path(code_root) / "artifacts" / "candidates").glob("*/candidate_manifest.json"))
    candidate_file = sorted(cands, key=lambda p: p.stat().st_mtime)[-1]

# Free any memory held by prior imports or checks before running gate
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=== Telemetry Prior to Gate Run ===")
try:
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print(f"nvidia-smi check skipped: {e}")

from scripts.run_gpu_gate import run_gpu_gate
report = run_gpu_gate(
    stage="kaggle_t4x2",
    candidate_path=str(candidate_file),
    data_dir=paths["runtime_root"],
    output_dir="/kaggle/working",
)
print("Gate execution completed with report status:", report.status)


In [ ]:
# Cell 7: Export Evidence Bundle & Verify Gate Report
import json, os
report_path = "/kaggle/working/kaggle_t4x2_report.json"
legacy_report_path = "/kaggle/working/kaggle_smoke_report.json"
if not os.path.exists(report_path):
    report_path = legacy_report_path

with open(report_path, "r", encoding="utf-8") as f:
    report = json.load(f)

print(f"Gate Status: {report.get('status')}")
print(f"Candidate ID: {report.get('candidate_id')}")
assert report.get("status") == "PASS", f"Kaggle dual-T4 smoke gate failed: {report}"
print("Kaggle Dual-T4 Gate Verification: PASS")
